# 3주차 — 변수 freeze 최종 확정 + XGBoost 분위수 회귀

> 계획서 v5.1 §9 3주차 작업 · 검증 포인트: **freeze 산출물 4건 완성 + 베이스라인 3개 비교표**

## 본 노트북의 목표

| 작업 | 계획서 근거 | 산출물 |
|------|------------|--------|
| 1. 변수 freeze 최종 확정 (9 → 8, kospi 제외) | §3.2(a) 산출물 4건 | `docs/freeze_final_w3.md` |
| 2. 5주차 ablation 대상 사전 명시 (환율 필수 + 후보) | §3.2(b) | `docs/ablation_plan_w3.md` |
| 3. XGBoost 분위수 회귀 (q=0.05, 0.5, 0.95) | §5.1 | `models/xgb_q*.json`, `data/processed/xgb_predictions_w3.csv` |
| 4. Naive·ARIMA·XGBoost 비교 (3주차 검증 포인트) | §9 | `reports/baseline_results_w3.csv` |

## 입력 (2주차 산출물)

- `data/processed/features_v1_candidate.csv` — 9 freeze + 타겟
- `data/processed/features_with_lags_v1.csv` — Lag/Rolling + 라벨
- `models/scaler_robust_train.pkl` — train-fit RobustScaler
- `reports/baseline_results_w2.csv` — Naive/Momentum/Random/ARIMA
- `docs/feature_validation_w2.md` — 종합 점수표

## 후속 단계 (4주차 → `04_lstm_quantile.ipynb`)

다변량 LSTM 분위수 회귀 + LSTM-SHAP DeepExplainer.

---

## 0. 환경 설정

In [ ]:
# 의존성 자동 체크
import importlib.util, subprocess, sys
REQUIRED = {
    'xgboost':     'xgboost',
    'sklearn':     'scikit-learn',
    'matplotlib':  'matplotlib',
    'yaml':        'pyyaml',
}
for _name, _pip in REQUIRED.items():
    if importlib.util.find_spec(_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', _pip])
print('✅ 의존성 체크 완료')

import pickle
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 100

PROJECT_ROOT = Path.cwd().parent
DATA_DIR     = PROJECT_ROOT / 'data'
MODELS_DIR   = PROJECT_ROOT / 'models'
REPORT_DIR   = PROJECT_ROOT / 'reports'
FIG_DIR      = REPORT_DIR / 'figures'
DOCS_DIR     = PROJECT_ROOT / 'docs'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

with open(PROJECT_ROOT / 'configs' / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 180)
print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'xgboost {xgb.__version__}')

---

## 1. 2주차 산출물 로드

In [ ]:
# (1) 1주차 freeze 9개 + 타겟 (raw)
features_v1 = pd.read_csv(
    DATA_DIR / 'processed' / 'features_v1_candidate.csv',
    index_col='date', parse_dates=['date']
).sort_index()
TARGET = CONFIG['project']['target']
FROZEN_W1 = [c for c in features_v1.columns if c != TARGET]
print(f'1주차 freeze (9개): {FROZEN_W1}')

# (2) Lag/Rolling 적용된 feature + 라벨
features_with_lags = pd.read_csv(
    DATA_DIR / 'processed' / 'features_with_lags_v1.csv',
    index_col='date', parse_dates=['date']
).sort_index()
print(f'features_with_lags shape: {features_with_lags.shape}')

# (3) train-fit Scaler
with open(MODELS_DIR / 'scaler_robust_train.pkl', 'rb') as f:
    scaler_pkg = pickle.load(f)
scaler = scaler_pkg['scaler']
FEATURE_COLS_W2 = scaler_pkg['feature_cols']  # W2 의 모든 lag/rolling 컬럼
SPLIT = scaler_pkg['split']
print(f'Scaler: 학습된 feature 수 = {len(FEATURE_COLS_W2)}')

# (4) W2 베이스라인 결과
baseline_w2 = pd.read_csv(REPORT_DIR / 'baseline_results_w2.csv')
print(f'\nW2 베이스라인 ({len(baseline_w2)} rows):')
print(baseline_w2.to_string(index=False))

---

## 2. 변수 freeze 최종 확정 (9 → 8, **kospi 제외**)

### 결정 근거 (`docs/feature_validation_w2.md` §4 종합 점수)

| 점수 | 변수 | 처리 |
|------|------|------|
| 3 | `kr_treasury_3y`, `us_treasury_10y`, `us_breakeven_10y`, `dxy` | ✅ 강력 채택 |
| 2 | `kr_base_rate`, `us_fed_funds`, `vix`, `sp500` | ✅ 채택 (정책/위험 채널) |
| **1** | **`kospi`** | **❌ 제외** (Granger p=0.57) |

### `kospi` 제외 사유
- **Granger p=0.5705** (best_lag=1) → 통계적으로 타겟 선행 못함
- **상관 \|r\|=0.043** → 거의 무관
- 위험자산 채널은 **`sp500`(글로벌)** + **`vix`(변동성)** 2개로 이미 커버
- **5주차 ablation 후보**로 명시 (kospi re-add) → 도메인 가설 vs 데이터 정량 비교

→ **8개 freeze**: `kr_treasury_3y`, `kr_base_rate`, `us_treasury_10y`, `us_fed_funds`, `us_breakeven_10y`, `vix`, `sp500`, `dxy`

In [ ]:
FROZEN_W3 = [v for v in FROZEN_W1 if v != 'kospi']
EXCLUDED_W3 = {'kospi': 'Granger p=0.57 (best_lag=1) → 타겟 선행 못함. sp500+vix 가 위험 채널 커버. 5주차 ablation 후보.'}
print(f'W3 freeze: {len(FROZEN_W3)}개')
for v in FROZEN_W3:
    print(f'  ✅ {v}')
for v, reason in EXCLUDED_W3.items():
    print(f'  ❌ {v}: {reason}')

# Lag/Rolling 컬럼 필터링 — base_name(컬럼) 이 FROZEN_W3 또는 라벨에 속해야 keep
def base_name(col: str) -> str:
    return col.split('__')[0]

KEEP_BASES = set(FROZEN_W3)  # 라벨 컬럼은 lag/rolling 없으므로 별도 처리 불필요
FEATURE_COLS_W3 = [c for c in features_with_lags.columns
                   if base_name(c) in KEEP_BASES]
print(f'\nW3 feature 수 (raw + lag + rolling): {len(FEATURE_COLS_W3)}')
print(f'  - raw 8 + lag 8×5 + rolling 8×6 = 8+40+48 = 96 예상')

# Scaler 인덱스 매핑 — W2 scaler 의 학습 컬럼 중 W3 에 해당하는 것만 골라 transform
scaler_idx_w3 = [FEATURE_COLS_W2.index(c) for c in FEATURE_COLS_W3]
print(f'Scaler 컬럼 매핑: {len(scaler_idx_w3)}개 (W2 scaler 재사용, refit 안 함 — CL-02)')

# freeze_final_w3.md 저장
lines = ['# 3주차 변수 freeze 최종 확정 (8개)', '',
         '> 계획서 v5.1 §3.2(a) 산출물 4 — `feature_validation_w2.md` (산출물 1~3) 기반 결정 문서.',
         f'> 생성: notebooks/03_freeze_xgboost.ipynb · 2026-05-02', '',
         '## 1. 채택 (8개)', '',
         '| # | 변수 | W2 점수 | 채택 사유 |',
         '|---|------|---------|-----------|']
RATIONALE_W3 = {
    'kr_treasury_3y':   '국내 단기 금리 (장단기 스프레드 신호)',
    'kr_base_rate':     '한국 통화정책 전이 채널 (lag 1 강제)',
    'us_treasury_10y':  '한미 금리 동조성 (Granger lag 3)',
    'us_fed_funds':     '미국 통화정책 기준 (lag 1)',
    'us_breakeven_10y': '기대인플레 (명목 = 실질 + 기대 항등식)',
    'vix':              '글로벌 위험 신호',
    'sp500':            'EDA SHAP 1위 (비선형/interaction)',
    'dxy':              'EM 자본유출 채널 (환율 부재 보완)',
}
SCORES_W2 = {'kr_treasury_3y':3, 'us_treasury_10y':3, 'us_breakeven_10y':3, 'dxy':3,
             'kr_base_rate':2, 'us_fed_funds':2, 'vix':2, 'sp500':2, 'kospi':1}
for i, v in enumerate(FROZEN_W3, 1):
    lines.append(f'| {i} | `{v}` | {SCORES_W2[v]} | {RATIONALE_W3[v]} |')
lines += ['', '## 2. 제외 (W3 단계 — 1개)', '',
          '| 변수 | W2 점수 | 제외 사유 |',
          '|------|---------|-----------|']
for v, reason in EXCLUDED_W3.items():
    lines.append(f'| `{v}` | {SCORES_W2[v]} | {reason} |')
lines += ['',
          '> **참고**: W1 단계에서 이미 다중공선성/약한 신호로 제외된 변수는 `VALIDATION_LOG.md` **#29** 참조:',
          '> - 한국 금리 패밀리 (`kr_treasury_5y`, `kr_treasury_1y`, `kr_corp_aa3y`, `kr_cd_91d`) — \\|r\\|>0.7 다중공선성, `kr_treasury_3y` 1개만 채택',
          '> - 월별 인플레/실물 (`kr_cpi`, `kr_cpi_core`, `kr_ppi`, `kr_industrial_prod`, `kr_mfg_bsi_outlook`, `us_cpi`) — 일별 Δy 와 거의 무관',
          '> - `us_treasury_2y` (us_treasury_10y 다중공선성), `us_credit_spread` (vix 다중공선성), `wti_oil` (약한 신호)',
          '>',
          '> 즉 W3 의 8개 freeze = (광역 22 후보) − (W1 다중공선성·약한신호 13개) − (W2 검증 약점 1개=kospi).',
          '',
          '## 3. 거시경제 채널 일관성 (계획서 §3.1)', '',
          '- 정책: `kr_base_rate`, `us_fed_funds`',
          '- 인플레: `us_breakeven_10y`',
          '- 동조성: `us_treasury_10y`',
          '- 위험: `vix`',
          '- 자산: `sp500` (글로벌만, kospi 제거)',
          '- 글로벌달러: `dxy`',
          '- 한국 단기: `kr_treasury_3y`',
          '',
          '5개 채널 모두 커버. kospi 제거로 자산 채널이 글로벌 1개로 축소되지만 5주차 ablation 에서 재검증.',
          '']
freeze_path = DOCS_DIR / 'freeze_final_w3.md'
freeze_path.write_text('\n'.join(lines), encoding='utf-8')
print(f'\n💾 저장: {freeze_path.relative_to(PROJECT_ROOT)}')

---

## 3. 5주차 ablation 대상 사전 명시 (계획서 §3.2(b))

**(C 수정) 환율 ablation 1개 필수 + 추가 1~2개는 시간 여유 시 선택.**

### 사전 명시 — 5주차 결정 미루기 금지

| ID | 종류 | 변수 변화 | 가설 | 비교 metric |
|----|------|-----------|------|-------------|
| **A1** | 🔴 필수 | 8 + `krw_usd` → 9 | 환율 추가가 정부 개입 시점에 의미 있는 SHAP 기여 | Pinball/Coverage/Sharpness/RMSE + DM test |
| A2 | 🟠 선택 | 8 + `kospi` → 9 | 국내 위험자산 채널의 도메인 가설 | 동일 |
| A3 | 🟡 선택 | 8 + `kr_ppi` → 9 | SHAP 2위 (월별 ffill 가짜 신호 의심) 사후 검증 | 동일 |

### 음수 채택 원칙 (§3.2(c))
결과가 부정적이어도 그 음수 자체를 결과로 보고. "왜 효과가 없었는가" 를 §6.4 오류 분석 자료로 활용.

In [ ]:
lines = ['# 5주차 Ablation 계획서 (3주차 사전 명시)', '',
         '> 계획서 v5.1 §3.2(b) — 환율 1개 필수 + 추가 1~2개 선택.',
         '> 음수 채택 원칙 §3.2(c): 부정적 결과도 §6.4 오류 분석 자료로 보고.',
         f'> 생성: notebooks/03_freeze_xgboost.ipynb · 2026-05-02', '',
         '## 사전 명시된 Ablation 후보',  '',
         '| ID | 우선순위 | 변경 | 가설 | 평가 metric |',
         '|----|---------|------|------|-------------|',
         '| A1 | 🔴 필수 | base 8 + `krw_usd` → 9 | 환율 추가가 정부 개입 시점 SHAP 기여 | Pinball / Coverage / Sharpness / RMSE / DM(HLN+Bonferroni) |',
         '| A2 | 🟠 선택 | base 8 + `kospi` → 9 | 국내 위험자산 채널 가설 | 동일 |',
         '| A3 | 🟡 선택 | base 8 + `kr_ppi` → 9 | SHAP 2위(월별 ffill 가짜 신호?) 사후 검증 | 동일 |',
         '',
         '## 평가 baseline',  '',
         '- **Base 모델**: 본 3주차 XGBoost 분위수 회귀 (8 vars, q=[0.05, 0.5, 0.95])',
         '- **비교 모델**: 5주차 LSTM 분위수 회귀 (튜닝 후 베스트)',
         '- 각 ablation 마다 동일 분할/시드/하이퍼파라미터로 학습 후 비교',
         '',
         '## 데이터 준비 절차 (5주차 진입 시)',  '',
         '### A1 (환율) — 새 변수 추가',
         '1. `data/interim/wide_daily_filled.csv` 에서 `krw_usd` 컬럼 추출',
         '2. `02b_preprocess_baseline.ipynb` §5~§6 동일 패턴으로 lag/rolling feature 생성:',
         '   - `krw_usd__lag{1,5,10,20,30}` (5개)',
         '   - `krw_usd__rmean{5,10,20}`, `krw_usd__rstd{5,10,20}` (6개)',
         '3. RobustScaler **train 만 재fit** (CL-02) — 9 vars × 11 = 99 컬럼 학습 → `models/scaler_robust_train_a1.pkl` 별도 저장',
         '4. CL-10 (환율 분리 원칙) ablation 모델에는 명시적으로 사용한다고 docs/ 에 기록',
         '',
         '### A2 (kospi re-add) — 기존 컬럼 재포함',
         '1. `features_with_lags_v1.csv` 에 이미 `kospi__lag*`, `kospi__rmean*`, `kospi__rstd*` 존재',
         '2. W3 의 `KEEP_BASES` 에 `kospi` 만 추가 → 96 + 11 = 107 컬럼',
         '3. Scaler 재fit 불필요 — `scaler_robust_train.pkl` 의 W2 학습 컬럼 (108개) 중 인덱스 매핑만 변경',
         '',
         '### A3 (kr_ppi) — 월별 변수 추가',
         '1. `wide_daily_filled.csv` 에서 `kr_ppi` 추출 (이미 ffill 적용된 일별 인덱스)',
         '2. **CL-01 발표일 시프트 검증**: 익월 25일경 발표 → +1개월 lag 강제. 4주차 LSTM 진입 전 검증 노트북 별도 작성 권장',
         '3. lag/rolling feature 생성 후 RobustScaler 재fit → `scaler_robust_train_a3.pkl`',
         '',
         '## 결과 보고 형식 (5주차 채움)',  '',
         '| Ablation | Δ Pinball | Δ Coverage | Δ Sharpness | Δ RMSE | DM p-value |',
         '|----------|-----------|------------|-------------|--------|------------|',
         '| A1 환율 추가 | ? | ? | ? | ? | ? |',
         '| A2 kospi 추가 | ? | ? | ? | ? | ? |',
         '| A3 kr_ppi 추가 | ? | ? | ? | ? | ? |',
         '']
ablation_path = DOCS_DIR / 'ablation_plan_w3.md'
ablation_path.write_text('\n'.join(lines), encoding='utf-8')
print(f'💾 저장: {ablation_path.relative_to(PROJECT_ROOT)}')

---

## 4. 데이터 분할 + Scaler transform (8 vars × {raw + lag + rolling})

- W2 scaler 의 컬럼 중 W3 8개에 해당하는 인덱스만 사용 → **refit 금지** (CL-02)
- Train: 2010-2020 / Cal: 2021 / Val: 2022 / Test: 2023-2025

In [ ]:
def slice_period(df, period):
    s, e = SPLIT[period]
    return df.loc[s:e]

X_train_raw = slice_period(features_with_lags, 'train')[FEATURE_COLS_W3]
X_cal_raw   = slice_period(features_with_lags, 'cal')[FEATURE_COLS_W3]
X_val_raw   = slice_period(features_with_lags, 'val')[FEATURE_COLS_W3]
X_test_raw  = slice_period(features_with_lags, 'test')[FEATURE_COLS_W3]

# scaler.transform — W2 의 모든 컬럼으로 transform 후, W3 컬럼만 인덱싱
def scaler_transform_w3(df_raw):
    # 누락된 W2-only 컬럼은 0 으로 채워서 transform (어차피 W3 컬럼만 사용)
    full = pd.DataFrame(0.0, index=df_raw.index, columns=FEATURE_COLS_W2)
    for c in FEATURE_COLS_W3:
        full[c] = df_raw[c].values
    transformed = scaler.transform(full)
    return pd.DataFrame(transformed[:, scaler_idx_w3], index=df_raw.index, columns=FEATURE_COLS_W3)

X_train = scaler_transform_w3(X_train_raw)
X_cal   = scaler_transform_w3(X_cal_raw)
X_val   = scaler_transform_w3(X_val_raw)
X_test  = scaler_transform_w3(X_test_raw)

y_train = slice_period(features_with_lags, 'train')['delta_y_bp']
y_cal   = slice_period(features_with_lags, 'cal')['delta_y_bp']
y_val   = slice_period(features_with_lags, 'val')['delta_y_bp']
y_test  = slice_period(features_with_lags, 'test')['delta_y_bp']

for name, X, yy in [('train',X_train,y_train),('cal',X_cal,y_cal),('val',X_val,y_val),('test',X_test,y_test)]:
    print(f'{name:6s}  X.shape={X.shape}  y.shape={yy.shape}')

---

## 5. XGBoost 분위수 회귀 학습 (q=0.05, 0.5, 0.95)

- `objective='reg:quantileerror'`, `quantile_alpha=q` (XGBoost 1.7+)
- 3개 모델 독립 학습 → §7 sort 후처리로 monotonicity 보정
- Early stopping: val pinball loss
- Hyperparam 은 config.yaml `models.xgboost` 사용 (max_depth=6, n_est=500, lr=0.05)

In [ ]:
QUANTILES = [0.05, 0.5, 0.95]
XGB_PARAMS = {
    **CONFIG['models']['xgboost'],
    'random_state': 42,
    'verbosity': 0,
    'tree_method': 'hist',
}
print(f'XGBoost params: {XGB_PARAMS}')

models = {}
for q in QUANTILES:
    print(f'\n--- q={q} 학습 ---')
    m = xgb.XGBRegressor(
        objective='reg:quantileerror',
        quantile_alpha=q,
        n_estimators=XGB_PARAMS['n_estimators'],
        max_depth=XGB_PARAMS['max_depth'],
        learning_rate=XGB_PARAMS['learning_rate'],
        early_stopping_rounds=XGB_PARAMS['early_stopping_rounds'],
        random_state=XGB_PARAMS['random_state'],
        verbosity=XGB_PARAMS['verbosity'],
        tree_method=XGB_PARAMS['tree_method'],
    )
    m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    print(f'  best_iteration: {m.best_iteration}, best_score (val pinball): {m.best_score:.4f}')
    models[q] = m

---

## 6. 평가 — Pinball Loss / Coverage 90% / Sharpness

- **Pinball Loss** (분위수별): `mean(max(q·diff, (q-1)·diff))` 작을수록 좋음
- **Coverage 90%**: `mean(q05 ≤ y ≤ q95)` — 목표 0.9
- **Sharpness**: `mean(q95 - q05)` — 작을수록 정밀

In [ ]:
def pinball_loss(y_true, y_pred, q):
    diff = y_true - y_pred
    return float(np.mean(np.maximum(q * diff, (q - 1) * diff)))

def evaluate_quantile_model(models_dict, X, y, label=''):
    preds = {q: m.predict(X) for q, m in models_dict.items()}
    out = {'split': label}
    for q in QUANTILES:
        out[f'pinball_q{int(q*100):02d}'] = pinball_loss(y.values, preds[q], q)
    cov = float(np.mean((y.values >= preds[0.05]) & (y.values <= preds[0.95])))
    sharp = float(np.mean(preds[0.95] - preds[0.05]))
    out['coverage_90'] = cov
    out['sharpness_bp'] = sharp
    # 점추정 (q50) 의 RMSE/MAE
    err = y.values - preds[0.5]
    out['rmse_q50_bp'] = float(np.sqrt(np.mean(err ** 2)))
    out['mae_q50_bp']  = float(np.mean(np.abs(err)))
    return out, preds

rows = []
preds_by_split = {}
for sp_name, X, yy in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    row, preds = evaluate_quantile_model(models, X, yy, label=sp_name)
    rows.append(row)
    preds_by_split[sp_name] = preds
eval_df = pd.DataFrame(rows)
for c in eval_df.columns[1:]:
    eval_df[c] = eval_df[c].round(3)
print('=== XGBoost 분위수 회귀 평가 (raw, sort 전) ===')
print(eval_df.to_string(index=False))

---

## 7. Monotonicity 점검 + sort 후처리 (계획서 §5.3)

독립 학습된 분위수 모델은 `q05 > q50` 또는 `q50 > q95` (quantile crossing) 가능.
→ 행별로 정렬하는 단순 sort 후처리로 보정.

In [ ]:
def count_crossings(preds_dict):
    arr = np.column_stack([preds_dict[q] for q in QUANTILES])
    n_05_gt_50 = int(np.sum(arr[:, 0] > arr[:, 1]))
    n_50_gt_95 = int(np.sum(arr[:, 1] > arr[:, 2]))
    return n_05_gt_50, n_50_gt_95

def sort_quantiles(preds_dict):
    arr = np.column_stack([preds_dict[q] for q in QUANTILES])
    arr_sorted = np.sort(arr, axis=1)
    return {q: arr_sorted[:, i] for i, q in enumerate(QUANTILES)}

preds_sorted_by_split = {}
for sp in ['train', 'val', 'test']:
    n1, n2 = count_crossings(preds_by_split[sp])
    total = len(preds_by_split[sp][0.05])
    print(f'{sp:6s}  N={total:>5d}  q05>q50: {n1:>4d} ({n1/total*100:.1f}%)  q50>q95: {n2:>4d} ({n2/total*100:.1f}%)')
    preds_sorted_by_split[sp] = sort_quantiles(preds_by_split[sp])

# sort 후 재평가
rows_sorted = []
for sp_name, X, yy in [('train', X_train, y_train), ('val', X_val, y_val), ('test', X_test, y_test)]:
    preds = preds_sorted_by_split[sp_name]
    out = {'split': sp_name}
    for q in QUANTILES:
        out[f'pinball_q{int(q*100):02d}'] = pinball_loss(yy.values, preds[q], q)
    cov = float(np.mean((yy.values >= preds[0.05]) & (yy.values <= preds[0.95])))
    sharp = float(np.mean(preds[0.95] - preds[0.05]))
    err = yy.values - preds[0.5]
    out['coverage_90'] = cov
    out['sharpness_bp'] = sharp
    out['rmse_q50_bp'] = float(np.sqrt(np.mean(err ** 2)))
    out['mae_q50_bp']  = float(np.mean(np.abs(err)))
    rows_sorted.append(out)
eval_df_sorted = pd.DataFrame(rows_sorted)
for c in eval_df_sorted.columns[1:]:
    eval_df_sorted[c] = eval_df_sorted[c].round(3)
print('\n=== sort 후 재평가 ===')
print(eval_df_sorted.to_string(index=False))

---

## 8. 방향성 정확도 — q50 기반 (LSTM 55% 비교 기준)

Naive(Δŷ=0) 는 sign(0)=0 이라 측정 불가 → q50 (median 예측) sign 사용.
Random(50%) / Momentum / ARIMA / XGBoost(q50) 비교.

In [ ]:
def dir_acc(y_true, y_pred):
    mask = (np.sign(y_pred) != 0) & (np.sign(y_true) != 0)
    if mask.sum() == 0:
        return float('nan')
    return float((np.sign(y_pred[mask]) == np.sign(y_true[mask])).mean())

xgb_dir = {}
for sp_name, yy in [('val', y_val), ('test', y_test)]:
    pred50 = preds_sorted_by_split[sp_name][0.5]
    da = dir_acc(yy.values, pred50)
    xgb_dir[sp_name] = da
    print(f'XGBoost(q50) {sp_name}: dir_acc = {da:.3f}')

print(f'\n📌 LSTM 목표 (계획서 §7): 방향성 ≥ 0.55')
print(f'   - XGBoost(q50) test {xgb_dir["test"]:.3f} 가 0.55 미만이면 LSTM 의 도전 목표 적정')
print(f'   - 0.55 초과면 XGBoost 가 이미 도달 → LSTM 은 Pinball/Coverage 에서 차별화 필요')

---

## 9. 베이스라인 3개 비교표 (3주차 검증 포인트)

Naive · ARIMA · **XGBoost(q50)** 의 RMSE/MAE/Dir_Acc 통합 비교.
(Momentum/Random 은 W2 추가 baseline 으로 함께 표시)

In [ ]:
# XGBoost row 추가
xgb_rows = []
for sp_name, yy in [('train', y_train), ('val', y_val), ('test', y_test)]:
    pred50 = preds_sorted_by_split[sp_name][0.5]
    err = yy.values - pred50
    rmse = float(np.sqrt(np.mean(err ** 2)))
    mae  = float(np.mean(np.abs(err)))
    da   = dir_acc(yy.values, pred50)
    xgb_rows.append({
        'model': 'XGBoost(q50)', 'split': sp_name,
        'RMSE_bp': round(rmse, 3), 'MAE_bp': round(mae, 3),
        'Dir_Acc': round(da, 3) if not np.isnan(da) else float('nan'),
    })
xgb_df = pd.DataFrame(xgb_rows)

# W2 baseline 과 통합
baseline_w3 = pd.concat([baseline_w2, xgb_df], ignore_index=True)
print('=== Naive · Momentum · Random · ARIMA · XGBoost(q50) 통합 비교 ===\n')
print(baseline_w3.to_string(index=False))

# 핵심 3개 (Naive, ARIMA, XGBoost) 만 dir_acc pivot
print('\n=== 핵심 3개 베이스라인 (val/test) ===')
core = baseline_w3[baseline_w3['model'].isin(['Naive (Δŷ=0)', 'ARIMA(1, 0, 1)', 'XGBoost(q50)'])]
core = core[core['split'].isin(['val', 'test'])]
print(core.to_string(index=False))

out_w3 = REPORT_DIR / 'baseline_results_w3.csv'
baseline_w3.to_csv(out_w3, index=False)
print(f'\n💾 저장: {out_w3.relative_to(PROJECT_ROOT)}')

---

## 10. 시각화 — 분위수 밴드 + 베이스라인 RMSE 비교

In [ ]:
# (1) Test 구간 분위수 밴드 시계열
fig, ax = plt.subplots(figsize=(14, 4.5))
test_idx = y_test.index
ps = preds_sorted_by_split['test']
ax.fill_between(test_idx, ps[0.05], ps[0.95], alpha=0.25, color='steelblue', label='90% 예측 구간')
ax.plot(test_idx, ps[0.5], color='steelblue', linewidth=0.8, label='q50 (median)')
ax.scatter(test_idx, y_test.values, s=4, color='black', alpha=0.5, label='실제 Δy')
ax.axhline(0, color='gray', linewidth=0.5)
ax.set_title(f'XGBoost 분위수 회귀 — Test 구간 (Coverage 90% target, 실제 {eval_df_sorted.iloc[2]["coverage_90"]:.3f})')
ax.set_ylabel('Δy (bp)')
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIG_DIR / 'w3_01_xgb_quantile_bands_test.png', dpi=120, bbox_inches='tight')
plt.show()

# (2) Val/Test RMSE 비교 — 핵심 3개
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
model_colors = {
    'Naive (Δŷ=0)':           'steelblue',
    'Momentum (Δŷ=Δy_lag1)':  'seagreen',
    'ARIMA(1, 0, 1)':         'darkorange',
    'XGBoost(q50)':           'crimson',
}
for ax, sp in zip(axes, ['val', 'test']):
    sub = baseline_w3[(baseline_w3['split']==sp) & baseline_w3['RMSE_bp'].notna()]
    bar_colors = [model_colors.get(m, 'gray') for m in sub['model']]
    ax.bar(sub['model'], sub['RMSE_bp'], color=bar_colors, alpha=0.8)
    ax.set_title(f'{sp.upper()} — RMSE (bp)')
    ax.set_ylabel('RMSE (bp)')
    for i, v in enumerate(sub['RMSE_bp'].values):
        ax.text(i, v, f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.tick_params(axis='x', rotation=20)
    ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(FIG_DIR / 'w3_02_baselines_compare.png', dpi=120, bbox_inches='tight')
plt.show()

---

## 11. 산출물 저장 — 모델 + 예측 CSV

In [ ]:
# (1) XGBoost 3개 모델
for q, m in models.items():
    p = MODELS_DIR / f'xgb_q{int(q*100):02d}.json'
    m.save_model(str(p))
    print(f'💾 {p.relative_to(PROJECT_ROOT)}')

# (2) Val/Test 예측 (sort 후 + 실제 + 점별 pinball loss → 6주차 DM test)
pred_rows = []
for sp_name, yy in [('val', y_val), ('test', y_test)]:
    ps = preds_sorted_by_split[sp_name]
    for date, true_v, p05, p50, p95 in zip(yy.index, yy.values, ps[0.05], ps[0.5], ps[0.95]):
        pred_rows.append({
            'date': date, 'split': sp_name,
            'y_true_bp': true_v,
            'q05': p05, 'q50': p50, 'q95': p95,
            'pinball_q05': max(0.05*(true_v-p05), -0.95*(true_v-p05)),
            'pinball_q50': max(0.5*(true_v-p50), -0.5*(true_v-p50)),
            'pinball_q95': max(0.95*(true_v-p95), -0.05*(true_v-p95)),
        })
pred_df = pd.DataFrame(pred_rows)
pred_path = DATA_DIR / 'processed' / 'xgb_predictions_w3.csv'
pred_df.to_csv(pred_path, index=False)
print(f'💾 {pred_path.relative_to(PROJECT_ROOT)}  ({len(pred_df):,}행)')

# (3) 분위수 평가 표 (raw + sorted)
eval_path = REPORT_DIR / 'xgb_quantile_eval_w3.csv'
eval_combined = pd.concat([
    eval_df.assign(stage='raw'),
    eval_df_sorted.assign(stage='sorted'),
], ignore_index=True)
eval_combined.to_csv(eval_path, index=False)
print(f'💾 {eval_path.relative_to(PROJECT_ROOT)}')

---

## 12. 다음 단계 (4주차 → `04_lstm_quantile.ipynb`)

1. **다변량 LSTM 분위수 회귀** — `(samples, 30 timesteps, 8 features)` 입력, q=[0.05, 0.5, 0.95] 동시 출력
2. **Pinball loss + monotonicity sort** 후처리
3. **LSTM-SHAP DeepExplainer** (실패 시 Permutation Importance 백업)

### 본 노트북 산출물 → 4주차 입력

- `models/xgb_q{05,50,95}.json` — XGBoost 3개 모델 (LSTM 비교 baseline)
- `data/processed/xgb_predictions_w3.csv` — Val/Test 예측 + 점별 pinball (6주차 DM test 입력)
- `reports/xgb_quantile_eval_w3.csv` — raw vs sorted 평가 비교
- `reports/baseline_results_w3.csv` — Naive/Momentum/Random/ARIMA/XGBoost(q50)
- `docs/freeze_final_w3.md` — 8 vars freeze 사유
- `docs/ablation_plan_w3.md` — 5주차 ablation A1(필수) + A2/A3(선택)

### TODO (이번 주 마감)

- [ ] §2 freeze 결정 (kospi 제외) → `VALIDATION_LOG.md` **#34** 기록
- [ ] §5 XGBoost 분위수 학습 결과 → `VALIDATION_LOG.md` **#35** 기록
- [ ] §3 ablation 계획 + §6.4 오류 분석 4축 매핑 → 발표 자료에 활용